### Differentiating Red Giant and Red Clump Stars Using Spectra and Machine Learning

### Resources
Bitmasks: https://www.sdss4.org/dr17/irspec/apogee-bitmasks/

Data: https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/

Filtering Values: https://ui.adsabs.harvard.edu/abs/2014ApJ...790..127B/abstract

In [3]:
from astropy.io import fits
import pandas as pd
import numpy as np

### PROCESSING RAW DATA
The raw data was in a format I was unfamiliar with and had a combination of scalar and array data types in columns. Therefore, we had to flatten the array columns to be able to store and access them properly in a Dataframe.

In [ ]:
filename = 'd:\Red_Giant_Classificaition_ML/allField-dr17.fits'
with fits.open(filename) as hdul:
    rec_data = hdul[1].data

    # Fixing endianness
    if rec_data.dtype.byteorder == '>':
        rec_data = rec_data.byteswap().newbyteorder()


In [82]:
# Adding scalar columns to Dataframe

scalar_cols = []
array_cols = []

for name in rec_data.columns.names:
    first_entry = rec_data[name][0]
    if np.shape(first_entry) == ():
        scalar_cols.append(name)
    else:
        array_cols.append(name)

df = pd.DataFrame({name: rec_data[name] for name in scalar_cols})


In [83]:
# Flattening array columns to add to Dataframe

for col in array_cols:
    arr = rec_data[col]
    nrows = arr.shape[0]
    flattened = arr.reshape(nrows, -1)
    col_shape = np.shape(arr[0])
    if len(col_shape) == 1:
        col_names = [f"{col}_{i}" for i in range(col_shape[0])]
    elif len(col_shape) == 2:
        col_names = [f"{col}_{i}_{j}" for i in range(col_shape[0]) for j in range(col_shape[1])]
    else:
        col_names = [f"{col}_{i}" for i in range(flattened.shape[1])]
    df_arrays = pd.DataFrame(flattened, columns=col_names)
    df = pd.concat([df, df_arrays], axis=1)


In [84]:
# Logistics

def to_native(series):
    try:
        arr = series.values
        dt = arr.dtype
    except AttributeError:
        return series

    if hasattr(dt, 'byteorder') and np.issubdtype(dt, np.number):
        if dt.byteorder == '>' or (dt.byteorder == '=' and not np.little_endian):
            arr = arr.byteswap().astype(dt.newbyteorder())
            return pd.Series(arr)
    return series

for col in df.columns:
    df[col] = to_native(df[col])


In [ ]:
# Saving initial processing data file

df.to_csv('D:\Red_Giant_Classificaition_ML/allstar_processed_fixed.csv', index=False)
print(df.head())


<>:3: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:3: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\hjmar\AppData\Local\Temp\ipykernel_50808\4279652213.py:3: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
  df.to_csv('D:\Astronomy-Final-Project/allstar_processed_fixed.csv', index=False)


                                  FILE           APOGEE_ID TELESCOPE  \
0               apStar-dr17-VESTA.fits               VESTA     apo1m   
1  apStar-dr17-2M00000002+7417074.fits  2M00000002+7417074    apo25m   
2  apStar-dr17-2M00000019-1924498.fits  2M00000019-1924498    apo25m   
3  apStar-dr17-2M00000032+5737103.fits  2M00000032+5737103    apo25m   
4  apStar-dr17-2M00000032+5737103.fits  2M00000032+5737103    apo25m   

   LOCATION_ID        FIELD ALT_ID         RA        DEC        GLON  \
0            1  calibration        -99.999001 -99.999001  292.219131   
1         5046       120+12   none   0.000103  74.285408  119.401807   
2         5071       060-75   none   0.000832 -19.413851   63.394122   
3         4424       116-04   none   0.001335  57.619530  116.065371   
4         4264        N7789   none   0.001335  57.619530  116.065371   

        GLAT  ...  GAIAEDR3_PHOT_BP_MEAN_MAG  GAIAEDR3_PHOT_RP_MEAN_MAG  \
0 -30.602919  ...                        NaN               

In [90]:
df.head()

,FILE,APOGEE_ID,TELESCOPE,LOCATION_ID,FIELD,ALT_ID,RA,DEC,GLON,GLAT,...,GAIAEDR3_PHOT_BP_MEAN_MAG,GAIAEDR3_PHOT_RP_MEAN_MAG,GAIAEDR3_DR2_RADIAL_VELOCITY,GAIAEDR3_DR2_RADIAL_VELOCITY_ERROR,GAIAEDR3_R_MED_GEO,GAIAEDR3_R_LO_GEO,GAIAEDR3_R_HI_GEO,GAIAEDR3_R_MED_PHOTOGEO,GAIAEDR3_R_LO_PHOTOGEO,GAIAEDR3_R_HI_PHOTOGEO
0,apStar-dr17-VESTA.fits,VESTA,apo1m,1,calibration,,-99.999001,-99.999001,292.219131,-30.602919,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,apStar-dr17-2M00000002+7417074.fits,2M00000002+7417074,apo25m,5046,120+12,none,0.000103,74.285408,119.401807,11.767414,...,13.2722,10.4956,-51.924702,0.365646,3234.574707,3027.029541,3464.556396,3268.601074,3115.693115,3507.732422
2,apStar-dr17-2M00000019-1924498.fits,2M00000019-1924498,apo25m,5071,060-75,none,0.000832,-19.413851,63.394122,-75.906397,...,12.5149,11.6597,17.880400,2.120880,2294.900146,1648.595459,3922.614014,888.348450,797.923157,970.810852
3,apStar-dr17-2M00000032+5737103.fits,2M00000032+5737103,apo25m,4424,116-04,none,0.001335,57.619530,116.065371,-4.564768,...,12.6500,11.6189,-19.195999,1.356420,762.945801,758.291321,767.333374,761.330017,756.740295,766.532227
4,apStar-dr17-2M00000032+5737103.fits,2M00000032+5737103,apo25m,4264,N7789,none,0.001335,57.619530,116.065371,-4.564768,...,12.6500,11.6189,-19.195999,1.356420,762.945801,758.291321,767.333374,761.330017,756.740295,766.532227


In [93]:
print(df.columns.to_list())

['FILE', 'APOGEE_ID', 'TELESCOPE', 'LOCATION_ID', 'FIELD', 'ALT_ID', 'RA', 'DEC', 'GLON', 'GLAT', 'J', 'J_ERR', 'H', 'H_ERR', 'K', 'K_ERR', 'SRC_H', 'WASH_M', 'WASH_M_ERR', 'WASH_T2', 'WASH_T2_ERR', 'DDO51', 'DDO51_ERR', 'IRAC_3_6', 'IRAC_3_6_ERR', 'IRAC_4_5', 'IRAC_4_5_ERR', 'IRAC_5_8', 'IRAC_5_8_ERR', 'IRAC_8_0', 'IRAC_8_0_ERR', 'WISE_4_5', 'WISE_4_5_ERR', 'TARG_4_5', 'TARG_4_5_ERR', 'WASH_DDO51_GIANT_FLAG', 'WASH_DDO51_STAR_FLAG', 'TARG_PMRA', 'TARG_PMDEC', 'TARG_PM_SRC', 'AK_TARG', 'AK_TARG_METHOD', 'AK_WISE', 'SFD_EBV', 'APOGEE_TARGET1', 'APOGEE_TARGET2', 'APOGEE2_TARGET1', 'APOGEE2_TARGET2', 'APOGEE2_TARGET3', 'APOGEE2_TARGET4', 'TARGFLAGS', 'SURVEY', 'PROGRAMNAME', 'NINST', 'NVISITS', 'COMBTYPE', 'COMMISS', 'SNR', 'SNREV', 'STARFLAG', 'STARFLAGS', 'ANDFLAG', 'ANDFLAGS', 'VHELIO_AVG', 'VSCATTER', 'VERR', 'RV_TEFF', 'RV_LOGG', 'RV_FEH', 'RV_ALPHA', 'RV_CARB', 'RV_CHI2', 'RV_CCFWHM', 'RV_AUTOFWHM', 'RV_FLAG', 'N_COMPONENTS', 'MEANFIB', 'SIGFIB', 'MIN_H', 'MAX_H', 'MIN_JK', 'MAX_JK'

In [ ]:
# Fixing endian issues

def fix_endian(series):
    arr = series.values
    dt = arr.dtype
    # Only fix if data type is numeric and big-endian
    if hasattr(dt, 'byteorder') and np.issubdtype(dt, np.number):
        if dt.byteorder == '>' or (dt.byteorder == '=' and not np.little_endian):
            arr = arr.byteswap().astype(dt.newbyteorder())
            return pd.Series(arr)
    return series

for col in df.columns:
    df[col] = fix_endian(df[col])


In [36]:
# Reloading saved dataframe
df = pd.read_csv('allstar_processed_fixed.csv')

### OBTAINING STUDY SAMPLE

In [42]:
# Good star filter
"""
From APOGEE bitmask documentation in reference 1, the STAR_BAD flag is set if essentially there are 
any other bad flags set for the star. The corresponding bit for this flag is bit 23. The telluric bitflags
are in APOGEE_TARGET2 and APOGEE2_TARGET2 and are bit 9 by the same documentation. COMMISS is a boolean column
indicating is the star is commissioning.
"""

STAR_BAD_BIT = 1 << 23
TELLURIC_BIT = 1 << 9

good_stars = (df["STARFLAG"] & STAR_BAD_BIT) == 0
non_telluric = (df["APOGEE_TARGET2"] & TELLURIC_BIT) == 0
non_telluric2 = (df["APOGEE2_TARGET2"] & TELLURIC_BIT) == 0
non_commissioning = df["COMMISS"] == 0
calibrated_surface_temp = df["RV_TEFF"] != -9999
calibrated_surface_grav = df["RV_LOGG"] != -9999
good_snr = df['SNR'] > 500

final_mask = good_stars & non_telluric & non_telluric2 & non_commissioning & calibrated_surface_temp & calibrated_surface_grav & good_snr

clean_df = df[final_mask]


In [43]:
print(clean_df)

                                       FILE           APOGEE_ID TELESCOPE  \
1       apStar-dr17-2M00000002+7417074.fits  2M00000002+7417074    apo25m   
12      apStar-dr17-2M00000317+5821383.fits  2M00000317+5821383    apo25m   
14      apStar-dr17-2M00000506+5656353.fits  2M00000506+5656353    apo25m   
15      apStar-dr17-2M00000506+5656353.fits  2M00000506+5656353    apo25m   
19      apStar-dr17-2M00000546+6152107.fits  2M00000546+6152107    apo25m   
...                                     ...                 ...       ...   
733794  apStar-dr17-2M23593396-1931485.fits  2M23593396-1931485    apo25m   
733852  apStar-dr17-2M23594841+5653269.fits  2M23594841+5653269    apo25m   
733857  apStar-dr17-2M23594955+1529189.fits  2M23594955+1529189    apo25m   
733858  apStar-dr17-2M23594955+1529189.fits  2M23594955+1529189    apo25m   
733877  apStar-dr17-2M23595490+5704387.fits  2M23595490+5704387    apo25m   

        LOCATION_ID        FIELD ALT_ID          RA        DEC        GLON 

In [44]:
# Filtering Red Giant and Red Clump Stars
"""
These filtering values are from the Bovy paper linked as the thrid reference.
"""

red_giant = (clean_df['RV_LOGG'].between(1.0, 3.5)) & (clean_df['RV_TEFF'].between(3500, 5200))
red_clump = (clean_df['RV_LOGG'].between(2.3, 2.5)) & (clean_df['RV_TEFF'].between(4500, 5000))

final_mask = red_giant | red_clump

data = clean_df[final_mask]


In [45]:
data

,FILE,APOGEE_ID,TELESCOPE,LOCATION_ID,FIELD,ALT_ID,RA,DEC,GLON,GLAT,...,GAIAEDR3_PHOT_RP_MEAN_MAG,GAIAEDR3_DR2_RADIAL_VELOCITY,GAIAEDR3_DR2_RADIAL_VELOCITY_ERROR,GAIAEDR3_R_MED_GEO,GAIAEDR3_R_LO_GEO,GAIAEDR3_R_HI_GEO,GAIAEDR3_R_MED_PHOTOGEO,GAIAEDR3_R_LO_PHOTOGEO,GAIAEDR3_R_HI_PHOTOGEO,parallax_snr
1,apStar-dr17-2M00000002+7417074.fits,2M00000002+7417074,apo25m,5046,120+12,none,0.000103,74.285408,119.401807,11.767414,...,10.49560,-51.92470,0.365646,3234.5747,3027.0295,3464.5564,3268.60100,3115.69300,3507.7324,14.656825
12,apStar-dr17-2M00000317+5821383.fits,2M00000317+5821383,apo25m,4424,116-04,none,0.013232,58.360649,116.219079,-3.839647,...,10.02780,-82.42390,0.320237,2577.4783,2455.5142,2708.9587,2599.15330,2482.81600,2716.9980,18.268911
14,apStar-dr17-2M00000506+5656353.fits,2M00000506+5656353,apo25m,-999,NGC7789_MGA,none,0.021113,56.943142,115.941040,-5.229802,...,10.40650,3.86782,0.690414,4507.9300,3204.2180,6286.7603,4396.21880,2438.39260,6019.0737,-0.387055
15,apStar-dr17-2M00000506+5656353.fits,2M00000506+5656353,apo25m,5922,NGC7789_btx,none,0.021113,56.943142,115.941040,-5.229802,...,10.40650,3.86782,0.690414,4507.9300,3204.2180,6286.7603,4396.21880,2438.39260,6019.0737,-0.387055
19,apStar-dr17-2M00000546+6152107.fits,2M00000546+6152107,apo25m,5045,116+00,none,0.022759,61.869644,116.918992,-0.401038,...,10.60350,-33.26460,0.507620,2539.6504,2461.3600,2642.7383,2590.07900,2493.90970,2710.1824,23.720874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
733790,apStar-dr17-2M23593308+6246339.fits,2M23593308+6246339,apo25m,5045,116+00,none,359.887838,62.776085,117.037735,0.499774,...,11.09580,-75.55440,0.334199,4311.5605,4060.3586,4576.8423,4242.69970,4023.02540,4464.8936,17.244206
733852,apStar-dr17-2M23594841+5653269.fits,2M23594841+5653269,apo25m,-999,NGC7789_MGA,none,359.951711,56.890812,115.893295,-5.273541,...,10.38460,-65.95200,0.299412,863.0326,853.5424,873.0935,869.95966,859.42584,883.1222,73.714216
733857,apStar-dr17-2M23594955+1529189.fits,2M23594955+1529189,apo25m,4548,105-45,none,359.956473,15.488585,105.025586,-45.580993,...,9.38465,12.97610,0.155913,1481.1115,1439.1489,1536.0101,1458.96770,1418.74150,1495.8407,31.947421
733858,apStar-dr17-2M23594955+1529189.fits,2M23594955+1529189,apo25m,2897,107-46_MGA,none,359.956473,15.488585,105.025586,-45.580993,...,9.38465,12.97610,0.155913,1481.1115,1439.1489,1536.0101,1458.96770,1418.74150,1495.8407,31.947421


In [46]:
# Parallax pruning

# Parallax signal to noise ratio
df['parallax_snr'] = df['GAIAEDR3_PARALLAX'] / df['GAIAEDR3_PARALLAX_ERROR']

# Define pruning mask
mask_parallax = (
    (df['GAIAEDR3_PARALLAX'].between(0, 1.5)) &
    (df['parallax_snr'] > 10)
)

data_pruned = data[mask_parallax]

print(f"Rows before pruning: {len(data)}")
print(f"Rows after pruning: {len(data_pruned)}")


Rows before pruning: 29069
Rows after pruning: 22266


C:\Users\hjmar\AppData\Local\Temp\ipykernel_36684\1529198742.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data_pruned = data[mask_parallax]


In [ ]:
data_pruned.to_csv('D:\Red_Giant_Classificaition_ML/cleaned_data.csv', index=False)

<>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\hjmar\AppData\Local\Temp\ipykernel_36684\3691470556.py:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
  data_pruned.to_csv('D:\Astronomy-Final-Project/cleaned_data.csv', index=False)
